In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis

### 환경설정 + vault 연결

In [0]:
import os
import sys
import json
from datetime import datetime

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
# vault 연결 후 ADLS OAuth 설정 (Databricks 전용)
vault.get_storage_client("datacopsadls")  # Spark conf에 OAuth 설정

# 설정 확인
spark = SparkSession.getActiveSession()
key = "fs.azure.account.auth.type.datacopsadls.dfs.core.windows.net"
print(f"[INFO] ADLS 인증 방식: {spark.conf.get(key, 'NOT SET')}")
# "OAuth" 가 나와야 정상

In [0]:
# 환경설정 셀 바로 다음에 추가
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession()

# FileSystem 캐시 초기화
spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.closeAll()
print("[OK] FileSystem 캐시 초기화 완료")

### Redis 연결 + 규칙 로드

In [0]:
from redis.cluster import RedisCluster, ClusterNode

redis_host     = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port     = int(vault.get_secret("redis-port"))

r = RedisCluster(
    startup_nodes=[ClusterNode(redis_host, redis_port)],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)
r.ping()
print(f"[OK] Redis 연결 완료: {redis_host}:{redis_port}")

# 01에서 감지된 도메인명으로 설정
DOMAIN_NAME     = "mediawiki_recentchange"
CACHE_KEY_RULES = f"gx_rules:{DOMAIN_NAME}"

BRONZE_PATH = "abfss://bronze@datacopsadls.dfs.core.windows.net/wikipedia/"
CONTAINER   = "silver"
ACCOUNT     = "datacopsadls"
BASE_PATH   = f"abfss://{CONTAINER}@{ACCOUNT}.dfs.core.windows.net"

# 규칙 없으면 시작 안 함
raw = r.get(CACHE_KEY_RULES)
if raw is None:
    raise RuntimeError(
        "Redis에 규칙이 없습니다. "
        "01_rules_generator를 먼저 실행하세요."
    )

current_rules = json.loads(raw)
print(f"[OK] 규칙 로드 완료")
print(f"  검증 규칙  : {len(current_rules.get('expectations', []))}개")
print(f"  이상치 탐지: {len(current_rules.get('anomaly_rules', []))}개")
print(f"  NULL 전략  : {len(current_rules.get('null_strategies', {}))}개")

### Spark 검증 + 이상치 탐지 함수 정의

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()

def validate_spark(df_spark, rules: dict) -> list:
    """Spark DataFrame 그대로 검증 — pandas 변환 없음"""
    results = []
    slim_profile = rules.get("slim_profile", {})

    for exp in rules.get("expectations", []):
        col      = exp.get("column")
        exp_type = exp.get("expectation_type")
        kwargs   = exp.get("kwargs", {})
        severity = exp.get("severity", "warning")

        if col not in df_spark.columns:
            continue

        try:
            if exp_type == "expect_column_values_to_not_be_null":
                if "null_when" in slim_profile.get(col, {}):
                    passed, fail_count = True, 0
                else:
                    fail_count = df_spark.filter(F.col(col).isNull()).count()
                    passed = fail_count == 0

            elif exp_type == "expect_column_values_to_be_in_set":
                allowed = kwargs.get("value_set", [])
                if allowed:
                    fail_count = df_spark.filter(
                        F.col(col).isNotNull() & ~F.col(col).isin(allowed)
                    ).count()
                    passed = fail_count == 0
                else:
                    passed, fail_count = True, 0
            else:
                passed, fail_count = True, 0

        except Exception:
            passed, fail_count = True, 0

        results.append({
            "column": col, "rule": exp_type,
            "severity": severity, "passed": passed, "fail_count": fail_count
        })
    return results


def detect_anomalies_spark(df_spark, anomaly_rules: list):
    """이상치 탐지 — Spark DataFrame 그대로 처리"""
    df_result = df_spark.withColumn("_anomaly_flags", F.lit(""))

    for rule in anomaly_rules:
        name      = rule.get("name")
        columns   = rule.get("columns", [])
        method    = rule.get("method")
        threshold = rule.get("threshold")
        col       = columns[0] if columns else None

        if not col or col not in df_spark.columns:
            continue

        try:
            if method == "zscore":
                stats = df_spark.select(
                    F.mean(col).alias("mean"),
                    F.stddev(col).alias("std")
                ).first()
                mean, std = stats["mean"], stats["std"]
                t = threshold or 3
                if std and std > 0:
                    df_result = df_result.withColumn(
                        "_anomaly_flags",
                        F.when(
                            F.abs((F.col(col) - mean) / std) > t,
                            F.concat(F.col("_anomaly_flags"), F.lit(f",{name}"))
                        ).otherwise(F.col("_anomaly_flags"))
                    )

            elif method == "iqr":
                q1, q3 = df_spark.approxQuantile(col, [0.25, 0.75], 0.05)
                iqr = q3 - q1
                t   = threshold or 1.5
                df_result = df_result.withColumn(
                    "_anomaly_flags",
                    F.when(
                        (F.col(col) < q1 - t * iqr) | (F.col(col) > q3 + t * iqr),
                        F.concat(F.col("_anomaly_flags"), F.lit(f",{name}"))
                    ).otherwise(F.col("_anomaly_flags"))
                )

            elif method == "frequency":
                freq  = df_spark.groupBy(col).count()
                stats = freq.select(
                    F.mean("count").alias("mean"),
                    F.stddev("count").alias("std")
                ).first()
                mean, std = stats["mean"], stats["std"]
                t = threshold or 3
                if std and std > 0:
                    high_freq = freq.filter(
                        F.col("count") > mean + t * std
                    ).select(col).rdd.flatMap(lambda x: x).collect()
                    if high_freq:
                        df_result = df_result.withColumn(
                            "_anomaly_flags",
                            F.when(
                                F.col(col).isin(high_freq),
                                F.concat(F.col("_anomaly_flags"), F.lit(f",{name}"))
                            ).otherwise(F.col("_anomaly_flags"))
                        )

        except Exception as e:
            print(f"  [WARN] {name} 탐지 실패: {e}")

    df_silver     = df_result.filter(F.col("_anomaly_flags") == "").drop("_anomaly_flags")
    df_quarantine = df_result.filter(F.col("_anomaly_flags") != "") \
                             .withColumnRenamed("_anomaly_flags", "_quarantine_reason")

    return df_silver, df_quarantine

# ── raw_json 파싱 ─────────────────────────────────────────
def parse_raw_json(df_spark):
    from pyspark.sql.functions import from_json, col
    from pyspark.sql.types import MapType, StringType

    df_parsed = df_spark.withColumn(
        "parsed", from_json(col("raw_json"), MapType(StringType(), StringType()))
    )

    # 전체 배치에서 등장하는 모든 키 합집합으로 추출
    all_keys = (
        df_parsed
        .select(F.explode(F.map_keys("parsed")).alias("key"))
        .distinct()
        .rdd.flatMap(lambda x: x)
        .collect()
    )

    if not all_keys:
        print("  [WARN] raw_json 파싱 실패 - 키 없음")
        return df_spark

    for key in all_keys:
        df_parsed = df_parsed.withColumn(key, col("parsed")[key])

    cols_to_drop = ["parsed", "raw_json", "_source",
                    "kafka_timestamp", "_bronze_loaded_at", "_kafka_topic"]
    existing_drops = [c for c in cols_to_drop if c in df_parsed.columns]
    df_parsed = df_parsed.drop(*existing_drops)

    print(f"  [OK] JSON 파싱 완료: {len(all_keys)}개 컬럼 추출")
    return df_parsed


# ── NULL 처리 (Spark 버전) ────────────────────────────────
def apply_null_strategies_spark(df_spark, null_strategies: dict):
    """
    Redis에서 로드한 null_strategies를 Spark DataFrame에 적용
    AI가 생성한 전략대로 null 채우기/제거
    """
    drop_cols = []

    for col_name, strategy in null_strategies.items():
        if col_name not in df_spark.columns:
            continue
        s = strategy.get("strategy")

        if s == "drop":
            drop_cols.append(col_name)
        elif s == "allow":
            pass
        elif s == "fill_default":
            default_val = strategy.get("default_value", "")
            if default_val is not None:
                df_spark = df_spark.fillna({col_name: str(default_val)})
        elif s == "fill_mode":
            mode_row = (
                df_spark.filter(F.col(col_name).isNotNull())
                .groupBy(col_name).count()
                .orderBy(F.desc("count"))
                .limit(1).collect()
            )
            if mode_row:
                df_spark = df_spark.fillna({col_name: str(mode_row[0][col_name])})
        elif s == "fill_mean":
            mean_val = df_spark.select(F.mean(col_name)).collect()[0][0]
            if mean_val is not None:
                df_spark = df_spark.fillna({col_name: mean_val})
        elif s == "fill_median":
            median_val = df_spark.approxQuantile(col_name, [0.5], 0.05)
            if median_val:
                df_spark = df_spark.fillna({col_name: median_val[0]})
        elif s == "fill_forward":
            from pyspark.sql.window import Window
            window = Window.orderBy(F.lit(1)).rowsBetween(
                Window.unboundedPreceding, 0
            )
            df_spark = df_spark.withColumn(
                col_name,
                F.last(col_name, ignorenulls=True).over(window)
            )

    if drop_cols:
        existing_drop = [c for c in drop_cols if c in df_spark.columns]
        if existing_drop:
            before = df_spark.count()
            for c in existing_drop:
                df_spark = df_spark.filter(F.col(c).isNotNull())
            after = df_spark.count()
            print(f"  [drop] {before - after}행 제거 | 컬럼: {existing_drop}")

    return df_spark

print("[OK] Spark 검증 함수 정의 완료")

### process_batch 정의 + 스트리밍 시작

In [0]:
CHECKPOINT_PATH = f"{BASE_PATH}/_checkpoints/bronze2silver"

def process_batch(df_spark, epoch_id):
    """
    Structured Streaming foreachBatch 콜백
    30초마다 Bronze Delta에 새로 쌓인 파일을 받아서 처리
    흐름: 파싱 → NULL처리 → 검증 → 이상치분리 → 저장 → 로그
    """
    count = df_spark.count()
    if count == 0:
        return

    print(f"\n[BATCH {epoch_id}] {count}건 수신")

    # 1. raw_json 파싱 → Wikipedia 이벤트 컬럼으로 펼치기
    df_spark = parse_raw_json(df_spark)

    # 2. Redis에서 최신 규칙 로드 (매 배치마다 갱신)
    raw = r.get(CACHE_KEY_RULES)
    if raw is None:
        print(f"[BATCH {epoch_id}] 규칙 없음 → 01_rules_generator 먼저 실행 필요")
        return
    current_rules = json.loads(raw)

    # 3. NULL 처리 — AI가 생성한 전략대로 null 채우기/제거
    df_spark = apply_null_strategies_spark(
        df_spark, current_rules.get("null_strategies", {})
    )

    # 4. 구조 검증 — 컬럼별 기대값 검사 (not null, value set 등)
    results      = validate_spark(df_spark, current_rules)
    total        = len(results)
    passed_count = sum(1 for r in results if r["passed"])

    # 5. 이상치 탐지 + Silver/Quarantine 분리
    df_silver, df_quarantine = detect_anomalies_spark(
        df_spark, current_rules.get("anomaly_rules", [])
    )

    # 6. 추적 메타 컬럼 추가
    processed_at = datetime.utcnow().isoformat() + "Z"
    run_ts       = datetime.utcnow().strftime("%H%M%S")
    today        = datetime.utcnow().strftime("%Y-%m-%d")

    df_silver = (
        df_silver
        .withColumn("_processed_at", F.lit(processed_at))
        .withColumn("_run_id",       F.lit(run_ts))
    )

    q_count = df_quarantine.count()
    if q_count > 0:
        df_quarantine = (
            df_quarantine
            .withColumn("_quarantine_ts", F.lit(processed_at))
            .withColumn("_processed_at",  F.lit(processed_at))
            .withColumn("_run_id",        F.lit(run_ts))
        )

    domain = current_rules.get("domain", "unknown_domain")

    # 7. Silver ADLS 저장 — date 파티션 + run_ts로 배치별 구분
    silver_path = f"{BASE_PATH}/{domain}/date={today}/run={run_ts}"
    df_silver.write.mode("overwrite").parquet(silver_path)

    # 8. Quarantine ADLS 저장
    if q_count > 0:
        q_path = f"{BASE_PATH}/quarantine/source={domain}/date={today}/run={run_ts}"
        df_quarantine.write.mode("overwrite").parquet(q_path)

    quality = round(passed_count / total * 100, 1) if total > 0 else 0
    print(f"[BATCH {epoch_id}] 완료 | Silver {df_silver.count()}행 | Quarantine {q_count}행 | 품질 {quality}%")

    # 9. GX 실행 로그 저장 — 품질 추이 모니터링용
    gx_log = {
        "run_id":          processed_at,
        "epoch_id":        epoch_id,
        "source":          domain,
        "date":            today,
        "total_rules":     total,
        "passed":          passed_count,
        "failed":          total - passed_count,
        "quality_score":   quality,
        "silver_rows":     df_silver.count(),
        "quarantine_rows": q_count,
    }
    log_path = f"{BASE_PATH}/logs/gx_runs/date={today}/run_{processed_at.replace(':', '-')}.json"
    log_json = json.dumps(gx_log, ensure_ascii=False, indent=2)
    spark.createDataFrame([(log_json,)], ["log"]) \
         .write.mode("overwrite").text(log_path)
    print(f"  [OK] GX 로그 저장: {log_path}")


# ── Structured Streaming 시작 ─────────────────────────────
# Bronze Delta를 스트림으로 읽음
# maxFilesPerTrigger=1: 한 번에 파일 1개씩 처리 (부하 조절)
# Checkpoint: 어디까지 처리했는지 기록 → 재시작해도 중복 없음
df_stream = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 1)
    .load(BRONZE_PATH)
)

query = (
    df_stream
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="30 seconds")
    .start()
)

print("[START] Structured Streaming 시작")
print(f"[INFO] Bronze    : {BRONZE_PATH}")
print(f"[INFO] Silver    : {BASE_PATH}/wikipedia_events/")
print(f"[INFO] Checkpoint: {CHECKPOINT_PATH}")

query.awaitTermination()